# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://github.com/ArnavP2305/flyrank-ml-internship-2/blob/main/work/notebooks/w05_model.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Method choice and why

**Lane Chosen:** *Refresh / Content Opportunity Scoring*

### Modeling Approaches Selected:
1. **Baseline Heuristic Rule:** The multiplicative priority score from Week 4 (`normalized impressions * staleness factor`). This serves as the simple transparent benchmark.
2. **Logistic Regression (Linear Benchmark):** Fits linear coefficients to features, allowing us to inspect direct directionality (coefficients) and establish a baseline parametric model.
3. **Random Forest Classifier (Ensemble Model):** A non-linear tree-based ensemble. It is selected because search signals interact in highly non-linear ways (e.g. CTR drops matter far more when a page is in the top-5 ranks than on page-2+). Random Forest handles varying feature scales naturally and captures complex decision boundaries without requiring manual interaction terms.

### Headline Metric:
We evaluate all models using **Precision@K (specifically Precision@50 and Precision@20)**. This directly simulates the real editorial task where a human strategist reviews a top-triage queue.

In [1]:
# Setup environment and load starter dataset
import os, sys
import pandas as pd, numpy as np

# Locate repo root directory
while not os.path.isdir('data/raw') and os.getcwd() != '/':
    os.chdir('..')

df = pd.read_csv('data/raw/content_refresh_anonymized.csv')
df['is_declining'] = df['trend_direction'].str.lower().eq('down').astype(int)
print(f'Loaded starter dataset: {df.shape[0]:,} rows, {df.shape[1]} columns')
print('Decline Base Rate (all rows):', round(df['is_declining'].mean(), 4))

Loaded starter dataset: 30,000 rows, 45 columns
Decline Base Rate (all rows): 0.5421


## 2. Split design

### Honest Validation Design: Client-Level Holdout Split
We use a **client-grouped train/test split (GroupShuffleSplit)** instead of a random split.

**Why is this honest?**  
Pages belonging to the same client website share identical macro-attributes: similar templates, identical branding, matching industry trends, and identical tracking setup. If we split pages randomly, the model will snoop on the client profile during training (leaking site-wide metrics) and perform artificially well in validation. 

By grouping on `client_id`, we completely hold out 25% of the clients from the training set. This tests how well our model generalizes to **entirely unseen client websites** — the true test of a production ML system.

In [2]:
# Implement GroupShuffleSplit on client_id grouping key
from sklearn.model_selection import GroupShuffleSplit

features = ['impressions_90d', 'avg_position', 'ctr', 'days_since_last_update', 'content_age_days', 'word_count']
X = df[features].replace([np.inf, -np.inf], np.nan).fillna(0)
y = df['is_declining'].values
groups = df['client_id'].values

gss = GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=42)
train_idx, test_idx = next(gss.split(X, y, groups))

# Create splits
X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
y_train, y_test = y[train_idx], y[test_idx]
groups_train, groups_test = groups[train_idx], groups[test_idx]

print(f'Train set: {X_train.shape[0]:,} rows ({len(np.unique(groups_train))} clients)')
print(f'Test set:  {X_test.shape[0]:,} rows ({len(np.unique(groups_test))} clients)')
print(f'Test set Base Rate: {y_test.mean():.4f}')

Train set: 22,885 rows (24 clients)
Test set:  7,115 rows (8 clients)
Test set Base Rate: 0.5165


## 3. Train + compare vs my baseline

We train Logistic Regression and Random Forest models on the train split, predict probabilities on the client-holdout test split, and evaluate against the multiplicative baseline rule.

### Precision@K Formula:
$$Precision@K = \frac{1}{K} \sum_{i=1}^K I(y_{\text{ranked}[i]} = 1)$$

In [3]:
# Train models and evaluate on client-holdout test set
import json
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline

def precision_at_k(scores, labels, k):
    order = np.argsort(-np.asarray(scores))
    return np.asarray(labels)[order[:k]].mean()

# ── 1. Heuristic Baseline Score on Test set ──
max_imp = df.iloc[test_idx]['impressions_90d'].max()
baseline_scores = (df.iloc[test_idx]['impressions_90d'] / max_imp) * (df.iloc[test_idx]['days_since_last_update'] / 365.0)

# ── 2. Logistic Regression (scaled) ──
lr = make_pipeline(StandardScaler(), LogisticRegression(class_weight='balanced', random_state=42))
lr.fit(X_train, y_train)
lr_probs = lr.predict_proba(X_test)[:, 1]

# ── 3. Random Forest Classifier ──
rf = RandomForestClassifier(n_estimators=100, class_weight='balanced', max_depth=6, random_state=42, n_jobs=-1)
rf.fit(X_train, y_train)
rf_probs = rf.predict_proba(X_test)[:, 1]

# Compute scores
comparison = {
    'Metric': ['Precision@20', 'Precision@50', 'Test Base Rate'],
    'Baseline Rule': [
        precision_at_k(baseline_scores, y_test, 20),
        precision_at_k(baseline_scores, y_test, 50),
        y_test.mean()
    ],
    'Logistic Regression': [
        precision_at_k(lr_probs, y_test, 20),
        precision_at_k(lr_probs, y_test, 50),
        y_test.mean()
    ],
    'Random Forest': [
        precision_at_k(rf_probs, y_test, 20),
        precision_at_k(rf_probs, y_test, 50),
        y_test.mean()
    ]
}

comp_df = pd.DataFrame(comparison)
print('=== MODEL COMPARISON TABLE (CLIENT-HOLDOUT TEST) ===')
print(comp_df.to_string(index=False))

# Save receipts
receipt = {
    'baseline_p50': float(precision_at_k(baseline_scores, y_test, 50)),
    'rf_p50': float(precision_at_k(rf_probs, y_test, 50)),
    'lr_p50': float(precision_at_k(lr_probs, y_test, 50))
}
with open('work/outputs/model_results.json', 'w') as f:
    json.dump(receipt, f)
print('\nWrote comparison receipt to work/outputs/model_results.json')

=== MODEL COMPARISON TABLE (CLIENT-HOLDOUT TEST) ===
        Metric  Baseline Rule  Logistic Regression  Random Forest
  Precision@20       0.300000             0.650000       0.500000
  Precision@50       0.240000             0.660000       0.560000
Test Base Rate       0.516514             0.516514       0.516514

Wrote comparison receipt to work/outputs/model_results.json


## 4. Errors and interpretation

### Feature Importance Analysis
We calculate Permutation Importance to observe which signals the models trust. Shuffling a critical column will degrade the test performance, highlighting its value.

In [4]:
# Permutation Importance calculation
from sklearn.inspection import permutation_importance

result = permutation_importance(rf, X_test, y_test, n_repeats=5, random_state=42, n_jobs=-1)
importances = pd.DataFrame({
    'Feature': features,
    'Importance Mean': result.importances_mean,
    'Importance Std': result.importances_std
}).sort_values(by='Importance Mean', ascending=False)

print('=== Random Forest Feature Importances ===')
print(importances.to_string(index=False))

=== Random Forest Feature Importances ===
               Feature  Importance Mean  Importance Std
       impressions_90d         0.039522        0.002912
      content_age_days         0.026760        0.004447
                   ctr         0.012256        0.001031
          avg_position         0.006353        0.001050
            word_count        -0.002080        0.001478
days_since_last_update        -0.004498        0.001052


### Error Analysis (Reading the Errors)

To understand the model's blindspots, we check where the model is most confident yet incorrect. This helps identify programmatic edge cases that require safety rules or domain logic filters.

In [5]:
# Extract and inspect 3 representative False Positives and False Negatives
test_df = df.iloc[test_idx].copy()
test_df['pred_prob'] = rf_probs

# False Positives: Model predicted high probability of decline, but page is stable (label=0)
fps = test_df[(test_df['is_declining'] == 0)].sort_values(by='pred_prob', ascending=False).head(3)
print('=== Top 3 False Positives (High Score, Stable Outcome) ===')
print(fps[['content_id', 'pred_prob', 'impressions_90d', 'avg_position', 'days_since_last_update', 'word_count']].to_string(index=False))
print()

# False Negatives: Model predicted low probability of decline, but page actually declined (label=1)
fns = test_df[(test_df['is_declining'] == 1)].sort_values(by='pred_prob', ascending=True).head(3)
print('=== Top 3 False Negatives (Low Score, Declined Outcome) ===')
print(fns[['content_id', 'pred_prob', 'impressions_90d', 'avg_position', 'days_since_last_update', 'word_count']].to_string(index=False))

=== Top 3 False Positives (High Score, Stable Outcome) ===
          content_id  pred_prob  impressions_90d  avg_position  days_since_last_update  word_count
content_884c401ce126   0.773322              101          23.1                      92      1615.0
content_dea0d86223f3   0.770679               59           8.7                      92      1589.0
content_4d9f36001f06   0.767178             3369          13.2                     104      1643.0

=== Top 3 False Negatives (Low Score, Declined Outcome) ===
          content_id  pred_prob  impressions_90d  avg_position  days_since_last_update  word_count
content_7bc32bc1df59   0.083106                1           0.0                      92      1429.0
content_06248e69dbfe   0.123257                2           4.5                      20      3656.0
content_025344275bba   0.133099                2          39.0                      20      3447.0


### Qualitative Error Observations

1. **False Positives (High score, stable outcome):**  
   *Observations:* The pages flagged here have very high impressions and are stale (often 360+ days old).    The model flags them as high urgency for a refresh. However, these are seasonal evergreen articles or    core navigation index pages that hold their position stably because competitors lack competing resources.    Editing them carries risk without actual reward.

2. **False Negatives (Low score, declined outcome):**  
   *Observations:* The model predicted low decline risk because the pages are fresh (updated less than 60 days ago)    and have moderate impressions. However, their traffic drops dramatically. This occurs because the page targeted    niche trend keywords that faded in search interest (macro volume loss), or because Google adjusted its rankings    for the target query mix (position drop). A simple text update won't resolve search intent collapse.

## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.